# Single-GPU CUDA checkpoint validation

Select **Runtime → Change runtime type → T4 GPU** if a free GPU is available, then run the cells in order. This notebook installs an isolated, locked environment; it does not upgrade your account or mount Drive.

The bounded FP32 experiment compares uninterrupted training with checkpoint reconstruction on the same GPU. It is not SIGKILL, runtime-disconnection, distributed-training, speedup, or model-quality evidence. No CUDA means a failure, never a CPU substitute.


In [ ]:
import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path


def checked(arguments, *, cwd=None, env=None, timeout=180):
    try:
        result = subprocess.run(
            arguments, cwd=cwd, env=env, check=True, capture_output=True, text=True, timeout=timeout
        )
    except subprocess.CalledProcessError as error:
        print((error.stdout or "")[-8000:])
        print((error.stderr or "")[-2000:])
        raise
    print(result.stdout[-8000:])
    if result.stderr:
        print(result.stderr[-2000:])
    return result.stdout


# Observe allocated hardware before downloading the locked CUDA environment.
hardware = checked(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total,compute_cap",
        "--format=csv,noheader",
    ]
)
assert hardware.strip(), "No NVIDIA GPU was observed"

In [ ]:
IMPLEMENTATION_REVISION = "4ffaf1da88f9092078d61ed3c801597eefcd0809"
assert re.fullmatch(r"[0-9a-f]{40}", IMPLEMENTATION_REVISION)
checked([sys.executable, "-m", "pip", "install", "uv==0.9.7"], timeout=900)
LAB_ROOT = Path(tempfile.mkdtemp(prefix="fttl-", dir="/content")) / "lab"
checked(
    [
        "git",
        "clone",
        "https://github.com/ajaykr0905/fault-tolerant-transformer-lab.git",
        str(LAB_ROOT),
    ],
    timeout=180,
)
checked(["git", "checkout", "--detach", IMPLEMENTATION_REVISION], cwd=LAB_ROOT)
assert checked(["git", "rev-parse", "HEAD"], cwd=LAB_ROOT).strip() == IMPLEMENTATION_REVISION
assert not checked(["git", "status", "--porcelain"], cwd=LAB_ROOT).strip()
RUN_ENV = dict(os.environ, CUBLAS_WORKSPACE_CONFIG=":4096:8")
checked(
    [sys.executable, "-m", "uv", "sync", "--frozen", "--extra", "test", "--python", "3.12"],
    cwd=LAB_ROOT,
    env=RUN_ENV,
    timeout=900,
)
PYTHON = str(LAB_ROOT / ".venv" / "bin" / "python")
# A fresh process owns CUDA initialization; a driver mismatch must stop here.
checked(
    [
        PYTHON,
        "-c",
        "import json; from fttl.cuda_runtime import prepare_cuda_execution; "
        "device, metadata = prepare_cuda_execution(); "
        "assert str(device) == 'cuda:0'; print(json.dumps(metadata, sort_keys=True))",
    ],
    cwd=LAB_ROOT,
    env=RUN_ENV,
)

In [ ]:
# Each test target gets a fresh process and its own CUDA initialization.
for target in [
    "tests/test_cuda_runtime.py",
    "tests/test_cuda_recovery.py",
    "tests/test_checkpoint_integrity.py::test_real_cuda_rng_checkpoint_is_weights_only_safe_and_cpu_loadable",
]:
    test_output = checked(
        [PYTHON, "-m", "pytest", "-q", "-rs", target], cwd=LAB_ROOT, env=RUN_ENV, timeout=180
    )
    assert "skipped" not in test_output, "Real CUDA regression tests were skipped"

DATASET = LAB_ROOT / ".cache" / "fttl" / "peps-v1"
checked(
    [
        PYTHON,
        "-m",
        "fttl.prepare_peps_cli",
        "--cache-dir",
        ".cache/downloads",
        "--output",
        str(DATASET),
    ],
    cwd=LAB_ROOT,
    env=RUN_ENV,
    timeout=180,
)

In [ ]:
OUTPUT = LAB_ROOT / "artifacts" / "colab-cuda-001"
assert not OUTPUT.exists(), "Use a fresh output directory for each experiment"
checked(
    [
        PYTHON,
        "-m",
        "fttl.cuda_recovery",
        "--config",
        "configs/peps-cpu.json",
        "--dataset-manifest",
        str(DATASET / "manifest.json"),
        "--output",
        str(OUTPUT),
        "--interruption-step",
        "2",
        "--max-eval-tokens",
        "4096",
    ],
    cwd=LAB_ROOT,
    env=RUN_ENV,
    timeout=180,
)
REPORT_PATH = OUTPUT / "cuda-recovery-report.json"
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
assert report["schema"] == "CudaRecoveryReportV1"
assert report["code_revision"] == IMPLEMENTATION_REVISION
assert report["execution"]["device"] == "cuda:0"
assert report["exact_equality"] is True and all(report["equality"].values())
assert report["steps"] == 6 and report["tokens_seen"] == 384
assert report["evaluation_device"] == "cpu"
assert report["memory"]["experiment_peak_allocated_bytes"] > 0
print(json.dumps(report, indent=2, sort_keys=True))

In [ ]:
# Download the small evidence report before Colab's temporary runtime disappears.
# Keep trusted-local .pt checkpoints out of Git and do not upload account data.
from google.colab import files

files.download(str(REPORT_PATH))